In [ ]:
# ============================================================
# 02_Preprocessing.ipynb
# Section 1 — Setup and Project Paths
# Purpose: mount Drive, define ALL paths, import preprocessing libraries.
# Same canonical block as 01_EDA.ipynb — keeps paths consistent across notebooks.
# ============================================================


import os, sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler   # used in Section 5
import joblib                                    # used to save the scaler

# --- The ONE root. Edit only this line if your Drive differs. ---
PROJECT_ROOT = os.path.join(DATA_DIR, "..")

# --- Every project folder ---
DATA_DIR             = os.path.join(PROJECT_ROOT, 'data')
EXPERIMENTS_DIR      = os.path.join(PROJECT_ROOT, 'experiments')
FIGURES_DIR          = os.path.join(PROJECT_ROOT, 'figures')
NOTEBOOKS_DIR        = os.path.join(PROJECT_ROOT, 'notebooks')
PAPERS_DIR           = os.path.join(PROJECT_ROOT, 'papers')
PROJECT_DECISION_DIR = os.path.join(PROJECT_ROOT, 'project_decision')
RESEARCH_JOURNAL_DIR = os.path.join(PROJECT_ROOT, 'research_journal')
RESULTS_DIR          = os.path.join(PROJECT_ROOT, 'results')
ROADMAP_DIR          = os.path.join(PROJECT_ROOT, 'RoadMap')
SRC_DIR              = os.path.join(PROJECT_ROOT, 'src')
THESIS_NOTES_DIR     = os.path.join(PROJECT_ROOT, 'thesis_notes')

# --- Where this notebook saves its outputs ---
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Make src/ importable so we can use our reusable functions ---
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# --- Verify every folder exists ---
for name, path in {
    "data": DATA_DIR, "data/processed": PROCESSED_DIR,
    "experiments": EXPERIMENTS_DIR, "figures": FIGURES_DIR,
    "notebooks": NOTEBOOKS_DIR, "papers": PAPERS_DIR,
    "project_decision": PROJECT_DECISION_DIR, "research_journal": RESEARCH_JOURNAL_DIR,
    "results": RESULTS_DIR, "RoadMap": ROADMAP_DIR, "src": SRC_DIR,
    "thesis_notes": THESIS_NOTES_DIR,
}.items():
    status = 'OK ' if os.path.isdir(path) else 'MISSING'
    print(f"{name:18s} {status}  {path}")

In [ ]:
# ============================================================
# 02_Preprocessing.ipynb
# Section 2 — Load FD001
# Purpose: load raw FD001 files via the reusable src/load_data.py.
# ============================================================

from load_data import load_cmapss_subset   # reusable loader (from 01_EDA)

# --- Load the FD001 subset from data/ ---
train, test, y_test = load_cmapss_subset(DATA_DIR, subset="FD001")

# --- Verify shapes match what EDA confirmed ---
print("train shape:", train.shape)                # expect (20631, 26)
print("test  shape:", test.shape)                 # expect (13096, 26)
print("y_test shape:", y_test.shape)              # expect (100, 1)
print("train engines:", train["unit"].nunique())  # expect 100
print("test  engines:", test["unit"].nunique())   # expect 100
print("missing values (train, test, y_test):",
      train.isna().sum().sum(), test.isna().sum().sum(), y_test.isna().sum().sum())

train.head()

In [ ]:
# ============================================================
# 02_Preprocessing.ipynb
# Section 3 — Apply Feature Removal
# Purpose: drop the 10 non-informative columns identified in EDA.
# ============================================================

# Force a fresh import in case src/preprocessing.py was edited this session
import importlib, preprocessing
importlib.reload(preprocessing)
from preprocessing import (
    drop_non_informative_columns,
    COLUMNS_TO_DROP_FD001,
    FINAL_FEATURES_FD001,
)

# --- Apply identical column drop to train and test ---
train_reduced = drop_non_informative_columns(train)
test_reduced  = drop_non_informative_columns(test)

# --- Verify ---
print("Columns dropped:", COLUMNS_TO_DROP_FD001)
print(f"\nBefore: train {train.shape}, test {test.shape}")
print(f"After:  train {train_reduced.shape}, test {test_reduced.shape}")
print(f"\nRemaining columns: {list(train_reduced.columns)}")

# Sanity check: the kept columns should be {unit, cycle} + 13 sensors
expected_cols = ["unit", "cycle"] + FINAL_FEATURES_FD001
actual_cols   = list(train_reduced.columns)
print(f"\nMatches expected layout: {actual_cols == expected_cols}")
print(f"Number of sensor features: {len(FINAL_FEATURES_FD001)}")

In [ ]:
# ============================================================
# 02_Preprocessing.ipynb
# Section 4 — Build the Two RUL Labels (TRAIN only)
# Purpose: add RUL_linear and RUL_piecewise125 to the training data.
# Test labels are NOT built here — they come from RUL_FD001.txt (Section 6).
# ============================================================

import importlib, preprocessing
importlib.reload(preprocessing)
from preprocessing import add_linear_rul, add_piecewise_rul

# --- Apply both labels to the TRAINING dataframe ---
train_labeled = add_linear_rul(train_reduced)
train_labeled = add_piecewise_rul(train_labeled, cap=125)

# --- Verify ---
print("Columns:", list(train_labeled.columns))
print(f"\nShape: {train_labeled.shape}  "
      f"(expect (20631, 17) — was 15, +2 label columns)\n")

# --- Sanity checks on the labels ---
print("RUL_linear stats:")
print(train_labeled["RUL_linear"].describe().round(1))

print("\nRUL_piecewise125 stats:")
print(train_labeled["RUL_piecewise125"].describe().round(1))

# --- Visual sanity: show engine 1's labels side by side ---
e1 = train_labeled[train_labeled["unit"] == 1][["cycle", "RUL_linear", "RUL_piecewise125"]]
print("\nEngine 1 — first 5 cycles (healthy phase, labels should differ):")
print(e1.head().to_string(index=False))
print("\nEngine 1 — last 5 cycles (failure phase, labels should match):")
print(e1.tail().to_string(index=False))

# --- Confirm the cap behavior across the fleet ---
n_clipped = (train_labeled["RUL_linear"] > 125).sum()
total_rows = len(train_labeled)
print(f"\nRows where the cap actually flattens RUL: "
      f"{n_clipped} / {total_rows} ({100*n_clipped/total_rows:.1f}%)")

In [ ]:
# ============================================================
# 02_Preprocessing.ipynb
# Section 5 — Min-Max Scaling (leakage-free)
# Purpose: scale 13 sensor features to [0,1]. Fit on TRAIN only.
# ============================================================

import importlib, preprocessing
importlib.reload(preprocessing)
from preprocessing import (
    fit_minmax_scaler, apply_minmax_scaler, FINAL_FEATURES_FD001
)
import joblib

# --- 1. Fit the scaler on TRAINING data only ---
# train_labeled is the (20631, 17) dataframe from Section 4
scaler = fit_minmax_scaler(train_labeled, FINAL_FEATURES_FD001)

# --- 2. Apply the SAME fitted scaler to both train and test ---
train_scaled = apply_minmax_scaler(train_labeled, scaler, FINAL_FEATURES_FD001)
test_scaled  = apply_minmax_scaler(test_reduced,  scaler, FINAL_FEATURES_FD001)

# --- 3. Verification ---
print("Train scaled — min/max per feature (should be [0, 1] by construction):")
print(train_scaled[FINAL_FEATURES_FD001].agg(["min", "max"]).T.round(4))

print("\nTest scaled — min/max per feature (may slightly exceed [0, 1] — see note below):")
print(test_scaled[FINAL_FEATURES_FD001].agg(["min", "max"]).T.round(4))

# --- 4. Confirm RUL labels were NOT scaled ---
print("\nRUL columns AFTER scaling (should be unchanged — same stats as Section 4):")
print(train_scaled[["RUL_linear", "RUL_piecewise125"]].describe().round(1).loc[["min", "max", "mean"]])

# --- 5. Save the scaler so model notebooks can reload it ---
scaler_path = os.path.join(PROCESSED_DIR, "minmax_scaler_FD001.joblib")
joblib.dump(scaler, scaler_path)
print(f"\nSaved fitted scaler to:\n{scaler_path}")

In [ ]:
# ============================================================
# 02_Preprocessing.ipynb
# Section 6 — Build the Test-Set Target
# Purpose: extract the last cycle of each test engine and attach RUL_true.
# ============================================================

import importlib, preprocessing
importlib.reload(preprocessing)
from preprocessing import build_test_last_cycle

# --- Build the 100-row evaluation ground truth ---
test_last_cycle = build_test_last_cycle(test_scaled, y_test)

# --- Verify ---
print(f"Shape: {test_last_cycle.shape}  (expect (100, 16) — 15 cols + RUL_true)")
print(f"\nEngines covered: {test_last_cycle['unit'].nunique()} (expect 100)")
print(f"\nRUL_true stats:")
print(test_last_cycle["RUL_true"].describe().round(1))

print("\nFirst 5 test engines (last-cycle row):")
print(test_last_cycle[["unit", "cycle", "RUL_true"]].head().to_string(index=False))

print("\nLast 5 test engines (last-cycle row):")
print(test_last_cycle[["unit", "cycle", "RUL_true"]].tail().to_string(index=False))

# --- Sanity check: test engine cycle lengths ---
test_cycle_lengths = test_scaled.groupby("unit")["cycle"].max()
print(f"\nTest engine truncation cycles — min: {test_cycle_lengths.min()}, "
      f"max: {test_cycle_lengths.max()}, median: {int(test_cycle_lengths.median())}")

In [ ]:
# ============================================================
# 02_Preprocessing.ipynb
# Section 7 — Save Processed Datasets
# Purpose: persist the three processed dataframes + scaler to data/processed/.
# These are loaded by model notebooks 03–06.
# ============================================================

# --- 1. Save the scaled training data (with both RUL labels) ---
train_path = os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet")
train_scaled.to_parquet(train_path, index=False)
print(f"Saved train  -> {train_path}")
print(f"   shape: {train_scaled.shape}, columns: {list(train_scaled.columns)}\n")

# --- 2. Save the scaled full test data (multi-cycle, no labels) ---
test_path = os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet")
test_scaled.to_parquet(test_path, index=False)
print(f"Saved test   -> {test_path}")
print(f"   shape: {test_scaled.shape}, columns: {list(test_scaled.columns)}\n")

# --- 3. Save the 100-row test evaluation ground truth ---
last_cycle_path = os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet")
test_last_cycle.to_parquet(last_cycle_path, index=False)
print(f"Saved last_cycle -> {last_cycle_path}")
print(f"   shape: {test_last_cycle.shape}, columns: {list(test_last_cycle.columns)}\n")

# --- 4. Confirm the scaler is still where Section 5 saved it ---
scaler_path = os.path.join(PROCESSED_DIR, "minmax_scaler_FD001.joblib")
print(f"Scaler at: {scaler_path}")
print(f"   exists: {os.path.exists(scaler_path)}")

# --- 5. Final sanity check: reload everything fresh and compare shapes ---
print("\n--- Reload check (proves the files are readable) ---")
train_reload = pd.read_parquet(train_path)
test_reload  = pd.read_parquet(test_path)
last_reload  = pd.read_parquet(last_cycle_path)
print(f"train reloaded: {train_reload.shape}  match: {train_reload.shape == train_scaled.shape}")
print(f"test  reloaded: {test_reload.shape}   match: {test_reload.shape  == test_scaled.shape}")
print(f"last  reloaded: {last_reload.shape}     match: {last_reload.shape  == test_last_cycle.shape}")

## Preprocessing Summary — FD001

This notebook took the raw FD001 files and the EDA-locked decisions from
`01_EDA.ipynb`, and produced a frozen, model-ready dataset stored in
`data/processed/`. All four model notebooks (03–06) load these files
directly — no model re-runs the preprocessing pipeline.

### What was done

1. **Loaded** raw FD001 via `src/load_data.py:load_cmapss_subset`.
2. **Dropped 10 non-informative columns** via
   `src/preprocessing.py:drop_non_informative_columns`:
   - op1, op2, op3 (near-constant; single operating condition)
   - s1, s5, s6, s10, s16, s18, s19 (zero / near-zero variance)
   - s14 (redundant with s9, r = 0.96)
3. **Built two RUL labels** on the training data via
   `add_linear_rul` and `add_piecewise_rul`:
   - `RUL_linear` = max_cycle − current_cycle (no cap)
   - `RUL_piecewise125` = same, clipped at 125 (Heimes 2008 convention)
   - 38.9 % of training rows flattened by the cap.
4. **Min-max scaling to [0, 1]** via `fit_minmax_scaler` + `apply_minmax_scaler`:
   - Fit on TRAINING features only (no leakage).
   - Applied identically to train and test.
   - RUL labels NOT scaled (kept in cycles).
5. **Built the test evaluation ground truth** via `build_test_last_cycle`:
   - For each of 100 test engines, kept the last available cycle.
   - Attached `RUL_true` from `RUL_FD001.txt`.

### Files in `data/processed/`

| File | Shape | Purpose |
|---|---|---|
| `train_FD001_processed.parquet` | (20631, 17) | Training: 13 scaled sensors + RUL_linear + RUL_piecewise125 |
| `test_FD001_processed.parquet`  | (13096, 15) | Full scaled test (multi-cycle), for LSTM/Chronos windows |
| `test_FD001_last_cycle.parquet` | (100, 16)   | Evaluation ground truth, 1 row per engine + RUL_true |
| `minmax_scaler_FD001.joblib`    | —           | Fitted MinMaxScaler for reuse |

### How the model notebooks will use these files

- **Linear Regression (03), XGBoost (04):** load `train_FD001_processed.parquet`
  → use 13 sensor columns as X, choose `RUL_linear` *or* `RUL_piecewise125` as y
  depending on the ablation variant. Evaluate against
  `test_FD001_last_cycle.parquet` (RUL_true).
- **LSTM (05), Chronos (06):** load `train_FD001_processed.parquet` and
  `test_FD001_processed.parquet`. Build sliding windows per engine.
  Evaluate the last-window prediction of each test engine against
  `test_FD001_last_cycle.parquet` (RUL_true).

### Fairness guarantee

All four models train on the same scaled features, with the same two label
options. All four models are evaluated against the same 100-row test ground
truth, with the same NASA test protocol (predict at last available cycle).
The only thing that differs between experiment runs is **(a)** the model and
**(b)** the RUL label variant. This gives the comparison 4 models × 2 labels
= 8 results, all directly comparable.

### Source basis

- Saxena et al. 2008 — FD001 structure, test protocol
- Heimes 2008 — piecewise-linear RUL with 125-cycle cap
- Asif et al. 2022 — preprocessing pipeline (correlation + low-variance filtering)
- Guilherme MSc 2025 — direct FD001 preprocessing reference
- scikit-learn — MinMaxScaler implementation